# Differential Equations — Session 11
## Section 3.2: Nonlinear Models

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. explain the density-dependent hypothesis;
2. derive and solve the logistic equation;
3. analyze logistic equilibria and concavity;
4. identify the maximum growth rate;
5. analyze logistic harvesting and its threshold;
6. compare logistic and Gompertz growth;
7. formulate a second-order chemical reaction using mass action;
8. identify the limiting reactant and long-term product amount.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–15 min | Density dependence and logistic derivation |
| 15–38 min | Logistic solution, equilibria, and inflection |
| 38–60 min | Parameter exploration and data interpretation |
| 60–76 min | Constant harvesting and threshold |
| 76–88 min | Mass-action chemical reaction |
| 88–90 min | Exit check |

The Gompertz comparison is an optional extension.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from matplotlib.patches import Rectangle, FancyArrowPatch, Circle
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def safe_solve(rhs, t_span, y0, points=700, **kwargs):
    t_eval = np.linspace(t_span[0], t_span[1], points)
    return solve_ivp(rhs, t_span, y0, t_eval=t_eval, **kwargs)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 3.2-A — Density-dependent model

A population model is **density dependent** when

$$
P'=P f(P),
$$

so the per-capita growth rate

$$
\frac{P'}{P}=f(P)
$$

depends on population size.

### Definition 3.2-B — Logistic equation

The logistic equation is

$$
P'=rP\left(1-\frac{P}{K}\right),
\qquad
r>0,\quad K>0,
$$

where $r$ is the low-density per-capita growth rate and $K$ is the carrying capacity.

### Theorem 3.2-C — Logistic solution

For $P(0)=P_0>0$,

$$
P(t)=
\frac{K}
{1+\left(\frac{K-P_0}{P_0}\right)e^{-rt}}.
$$

The equilibria are $P=0$ and $P=K$.

### Proposition 3.2-D — Stability and inflection

For positive populations:

- $P=0$ is unstable;
- $P=K$ is asymptotically stable;
- the growth rate is maximized at $P=K/2$;
- the maximum value of $P'$ is $rK/4$;
- a logistic curve changes concavity when $P=K/2$.

### Theorem 3.2-E — Constant harvesting threshold

For

$$
P'=rP\left(1-\frac{P}{K}\right)-h,
$$

positive equilibria exist only when

$$
h\le\frac{rK}{4}.
$$

At $h=rK/4$, the two positive equilibria merge at $P=K/2$. For larger harvest, no positive equilibrium exists.

### Definition 3.2-F — Gompertz equation

One form of the Gompertz model is

$$
P'=rP\ln\left(\frac{K}{P}\right).
$$

It also approaches $K$, but its growth curve is asymmetric.

### Principle 3.2-G — Law of mass action

For a second-order reaction with product amount $X(t)$,

$$
X'=k(\alpha-X)(\beta-X),
$$

where the factors represent the remaining reactant amounts after suitable stoichiometric scaling.

### Classroom Checkpoint — Maximum Logistic Growth

For

$$
P'=rP\left(1-\frac{P}{K}\right),
$$

at what population is the total growth rate largest?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. From exponential to logistic growth

Exponential growth assumes a constant per-capita rate:

$$
\frac{P'}{P}=r.
$$

Resource limitation suggests that this rate decreases with $P$. The simplest linear choice satisfying

$$
f(0)=r,
\qquad
f(K)=0
$$

is

$$
f(P)=r\left(1-\frac{P}{K}\right).
$$

In [ ]:
P = np.linspace(0, 1200, 500)
r, K = 0.5, 1000
per_capita = r*(1-P/K)
total_growth = P*per_capita

plt.plot(P, per_capita, label="per-capita growth rate")
plt.axhline(0, linestyle="--")
plt.axvline(K, linestyle=":")
plt.xlabel("population P")
plt.ylabel("rate")
plt.title("Density dependence in the logistic model")
plt.legend()
plt.show()

plt.plot(P, total_growth, label="total growth P'")
plt.axvline(K/2, linestyle="--", label="maximum at K/2")
plt.xlabel("population P")
plt.ylabel("P'")
plt.title("Logistic growth rate")
plt.legend()
plt.show()

## 2. Logistic solution and phase line

For

$$
P'=rP\left(1-\frac{P}{K}\right),
$$

the sign of $P'$ gives:

- increasing solutions for $0<P<K$;
- decreasing solutions for $P>K$;
- equilibrium at $P=0$ and $P=K$.

In [ ]:
def logistic_explorer(P0=100, r=0.4, K=1000, final_time=30):
    t = np.linspace(0, final_time, 700)
    A = (K-P0)/P0
    P = K/(1+A*np.exp(-r*t))

    plt.plot(t, P, linewidth=2)
    plt.axhline(K, linestyle="--", label="carrying capacity")
    plt.axhline(K/2, linestyle=":", label="inflection level")
    plt.scatter([0], [P0], s=70)
    plt.xlabel("time")
    plt.ylabel("population")
    plt.title("Interactive logistic growth")
    plt.legend()
    plt.show()

    if 0 < P0 < K:
        t_inflect = np.log((K-P0)/P0)/r
        print("Time at which P=K/2:", t_inflect)
    print("Maximum possible growth rate:", r*K/4)

if WIDGETS_AVAILABLE:
    interact(
        logistic_explorer,
        P0=FloatSlider(min=10, max=1500, step=10, value=100),
        r=FloatSlider(min=0.05, max=1.0, step=0.05, value=0.4),
        K=FloatSlider(min=200, max=2000, step=100, value=1000),
        final_time=IntSlider(min=10, max=80, step=5, value=30)
    )
else:
    logistic_explorer()

## 3. Concavity and the fastest-growth point

Differentiate

$$
P'=rP\left(1-\frac{P}{K}\right)
$$

with respect to time:

$$
P''=rP'\left(1-\frac{2P}{K}\right).
$$

For a growing solution with $0<P<K$:

- $P''>0$ when $P<K/2$;
- $P''<0$ when $P>K/2$.

Thus the graph changes from concave up to concave down at $P=K/2$.

In [ ]:
r, K, P0 = 0.35, 800, 40
t = np.linspace(0, 40, 700)
A = (K-P0)/P0
P = K/(1+A*np.exp(-r*t))
growth = r*P*(1-P/K)
t_star = np.log((K-P0)/P0)/r

plt.plot(t, P, label="P(t)")
plt.scatter([t_star], [K/2], s=80, label="inflection point")
plt.axhline(K/2, linestyle=":")
plt.xlabel("time")
plt.ylabel("population")
plt.legend()
plt.show()

plt.plot(t, growth)
plt.axvline(t_star, linestyle="--", label="maximum growth time")
plt.xlabel("time")
plt.ylabel("P'(t)")
plt.title("Growth rate peaks at the inflection point")
plt.legend()
plt.show()

## 4. Calibrating logistic growth from observations

If $K$ and $P_0$ are known and one later observation $P(t_1)$ is available, then

$$
P(t_1)=
\frac{K}
{1+\left(\frac{K-P_0}{P_0}\right)e^{-rt_1}}
$$

can be solved for $r$.

In [ ]:
K, P0, t1, P1 = 1200, 30, 5, 180
A = (K-P0)/P0
r_est = -(1/t1)*np.log((K/P1-1)/A)

print("Estimated r:", r_est)

t = np.linspace(0, 25, 600)
P = K/(1+A*np.exp(-r_est*t))
plt.plot(t, P)
plt.scatter([0, t1], [P0, P1], s=70)
plt.axhline(K, linestyle="--")
plt.xlabel("time")
plt.ylabel("population")
plt.title("Logistic parameter calibration")
plt.show()

## 5. Constant harvesting

Consider

$$
P'=rP\left(1-\frac{P}{K}\right)-h.
$$

The unharvested growth term is a parabola. Horizontal intersections with the harvest level $h$ are equilibria.

In [ ]:
def harvest_phase(r=0.5, K=1000, h=80, P0=700, final_time=40):
    P_grid = np.linspace(0, 1.2*K, 600)
    natural_growth = r*P_grid*(1-P_grid/K)
    hcrit = r*K/4

    plt.plot(P_grid, natural_growth, label="natural growth")
    plt.axhline(h, linestyle="--", label="harvest level")
    plt.axvline(K/2, linestyle=":")
    plt.xlabel("population P")
    plt.ylabel("rate")
    plt.title("Equilibria are intersections")
    plt.legend()
    plt.show()

    def rhs(t, P):
        return [r*P[0]*(1-P[0]/K)-h]

    sol = safe_solve(rhs, (0, final_time), [P0], rtol=1e-8, atol=1e-10)
    plt.plot(sol.t, sol.y[0], linewidth=2)
    plt.axhline(0, linestyle="--")
    plt.xlabel("time")
    plt.ylabel("population")
    plt.title("Population under constant harvest")
    plt.show()

    print("Critical harvest rK/4 =", hcrit)
    if h < hcrit:
        disc = 1-4*h/(r*K)
        P_low = K*(1-np.sqrt(disc))/2
        P_high = K*(1+np.sqrt(disc))/2
        print("Unstable equilibrium:", P_low)
        print("Stable equilibrium:", P_high)
    elif np.isclose(h, hcrit):
        print("One semistable equilibrium at K/2.")
    else:
        print("No positive equilibrium; the model predicts eventual collapse.")

if WIDGETS_AVAILABLE:
    interact(
        harvest_phase,
        r=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.5),
        K=FloatSlider(min=200, max=2000, step=100, value=1000),
        h=FloatSlider(min=0, max=300, step=10, value=80),
        P0=FloatSlider(min=10, max=1800, step=10, value=700),
        final_time=IntSlider(min=10, max=100, step=5, value=40)
    )
else:
    harvest_phase()

## Optional extension — Logistic versus Gompertz growth

Both models approach the same carrying capacity, but their shapes and inflection levels differ.

In [ ]:
K, P0, r = 1000, 20, 0.35
t = np.linspace(0, 30, 700)

logistic = K/(1+((K-P0)/P0)*np.exp(-r*t))
gompertz = K*np.exp(np.log(P0/K)*np.exp(-r*t))

plt.plot(t, logistic, label="logistic")
plt.plot(t, gompertz, linestyle="--", label="Gompertz")
plt.axhline(K, linestyle=":")
plt.xlabel("time")
plt.ylabel("population")
plt.title("Two saturating nonlinear growth models")
plt.legend()
plt.show()

## 6. Second-order chemical reactions

Suppose the scaled initial reactant amounts are $\alpha$ and $\beta$, and $X(t)$ is the product amount. The remaining reactants are

$$
\alpha-X,
\qquad
\beta-X.
$$

The law of mass action gives

$$
X'=k(\alpha-X)(\beta-X).
$$

The physically meaningful interval is

$$
0\le X\le\min(\alpha,\beta).
$$

The smaller initial reactant amount determines the limiting product.

In [ ]:
def reaction_explorer(alpha=60.0, beta=40.0, k=0.002, X0=0.0, final_time=80):
    def rhs(t, X):
        return [k*(alpha-X[0])*(beta-X[0])]

    sol = safe_solve(rhs, (0, final_time), [X0], rtol=1e-9, atol=1e-11)
    limit = min(alpha, beta)

    plt.plot(sol.t, sol.y[0], linewidth=2, label="product X(t)")
    plt.axhline(limit, linestyle="--", label="limiting amount")
    plt.xlabel("time")
    plt.ylabel("product amount")
    plt.title("Second-order reaction")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        reaction_explorer,
        alpha=FloatSlider(min=10, max=100, step=5, value=60),
        beta=FloatSlider(min=10, max=100, step=5, value=40),
        k=FloatSlider(min=0.0005, max=0.01, step=0.0005, value=0.002),
        X0=FloatSlider(min=0, max=20, step=1, value=0),
        final_time=IntSlider(min=20, max=200, step=10, value=80)
    )
else:
    reaction_explorer()

## Classroom Checkpoint — Exit Check

For

$$
P'=0.6P\left(1-\frac{P}{900}\right)-h,
$$

find the maximum sustainable constant harvest.

> Pause here. Let students commit to an answer before running the next cell.